In [10]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re
import time
import random

In [11]:
user_agents = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/119.0"
]
all_controllers = []
session = requests.Session()

In [12]:
page = 1
max_pages = 40

In [13]:
while page <= max_pages:
    print(f"Scraping Page {page}...")
    url = f"https://www.amazon.in/s?k=gaming+controller&page={page}"
    
    headers = {
        "User-Agent": random.choice(user_agents),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.google.com/",
        "Accept-Encoding": "gzip, deflate, br",
        "DNT": "1" # Do Not Track
    }

    try:
        response = session.get(url, headers=headers, timeout=20)
        
        if response.status_code == 503:
            wait_time = random.randint(30, 60) # Massive wait to "cool down" the IP
            print(f"503 detected. Cooling down for {wait_time} seconds...")
            time.sleep(wait_time)
            continue # Try the SAME page again after cooling down
            
        if response.status_code != 200:
            print(f"Failed with status {response.status_code}. Skipping page.")
            page += 1
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        cards = soup.find_all("div", {"class": "puisg-col-inner"})

        if not cards:
            print("No data found on page (possible captcha). Waiting...")
            time.sleep(20)
            continue

        for card in cards:
            try:
                name_tag = card.find("h2")
                if not name_tag: continue
                name = name_tag.text.strip()

                offered_tag = card.find("span", {"class": "a-price-whole"})
                actual_tag = card.find("span", {"class": "a-text-price"})
                if not offered_tag: continue
                
                off_price = int(re.sub(r'\D', '', offered_tag.text))
                
                # Check for Actual Price (MRP)
                act_price = off_price
                if actual_tag:
                    mrp_span = actual_tag.find("span", {"class": "a-offscreen"})
                    if mrp_span:
                        act_price = int(re.sub(r'\D', '', mrp_span.text))

                # Extra analytical columns
                brand = name.split(' ')[0]
                rating_tag = card.find("span", {"class": "a-icon-alt"})
                rating = rating_tag.text.split(" ")[0] if rating_tag else "0"
                

                all_controllers.append({
                    "Product_Name": name,
                    "Brand": brand,
                    "Offered_Price": off_price,
                    "Actual_Price": act_price,
                    "Rating": rating,
                    "Connectivity": "Wireless" if "wireless" in name.lower() else "Wired",
                    "Compatibility": "Universal" if "pc" in name.lower() else "Console"
                })
            except:
                continue

        print(f"Page {page} successful. Current Total: {len(all_controllers)}")
        page += 1 # Move to next page only if current page was successful
        time.sleep(random.uniform(5, 10)) # Human-like reading delay

    except Exception as e:
        print(f"Network error: {e}")
        time.sleep(10)

Scraping Page 1...
Page 1 successful. Current Total: 24
Scraping Page 2...
Page 2 successful. Current Total: 46
Scraping Page 3...
Page 3 successful. Current Total: 69
Scraping Page 4...
Page 4 successful. Current Total: 92
Scraping Page 5...
Page 5 successful. Current Total: 113
Scraping Page 6...
Page 6 successful. Current Total: 136
Scraping Page 7...
Page 7 successful. Current Total: 158
Scraping Page 8...
Page 8 successful. Current Total: 181
Scraping Page 9...
Page 9 successful. Current Total: 204
Scraping Page 10...
Page 10 successful. Current Total: 227
Scraping Page 11...
Page 11 successful. Current Total: 250
Scraping Page 12...
Page 12 successful. Current Total: 273
Scraping Page 13...
Page 13 successful. Current Total: 296
Scraping Page 14...
Page 14 successful. Current Total: 316
Scraping Page 15...
Page 15 successful. Current Total: 338
Scraping Page 16...
Page 16 successful. Current Total: 360
Scraping Page 17...
Page 17 successful. Current Total: 383
Scraping Page 18...

In [14]:
df = pd.DataFrame(all_controllers)

In [15]:
df.to_csv("gaming_controllers_dataset.csv", index=False)
print(f"Done! Collected {len(df)} records.")

Done! Collected 576 records.
